# Uni-MuMER - Kaggle 2xT4 + DagsHub

Train QLoRA, theo dõi metric trực tiếp bằng MLflow và lưu toàn bộ artifact lên DagsHub.

In [1]:
# 1. Cấu hình
import json
import os
import sys
import uuid
from pathlib import Path

from kaggle_secrets import UserSecretsClient

PROJECT_DIR = "/kaggle/working/test-unimer"
CONDA_DIR = "/kaggle/working/miniconda"
ENV_DIR = f"{CONDA_DIR}/envs/unimumer"
PYTHON = f"{ENV_DIR}/bin/python"

BASE_YAML_CONFIG = "train/Uni-MuMER-train.yaml"

# Nên để YAML runtime trong /kaggle/working để không ghi đè YAML gốc
RUNTIME_YAML_CONFIG = "/kaggle/working/runtime_Uni-MuMER-train.yaml"

YAML_CONFIG = BASE_YAML_CONFIG
NOTEBOOK_PATH = "uni-mumer-kaggle-dagshub v4.ipynb"
OUTPUT_DIR = "saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora"

DAGSHUB_USERNAME = "NhatPot"
DAGSHUB_REPO = "test-unimer"
EXPERIMENT_NAME = "Uni-MuMER-Qwen2.5-VL-3B"
RUN_UUID = uuid.uuid4().hex

# Cho notebook import được module nội bộ trong repo, ví dụ scripts.runtime_yaml
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Cho các lệnh subprocess cũng thấy project
os.environ["PYTHONPATH"] = PROJECT_DIR

# Lấy DagsHub token từ Kaggle Secrets
DAGSHUB_TOKEN = UserSecretsClient().get_secret("DAGSHUB_TOKEN")
if not DAGSHUB_TOKEN:
    raise RuntimeError("Kaggle Secret DAGSHUB_TOKEN is missing")

os.environ.update({
    "PROJECT_DIR": PROJECT_DIR,
    "CONDA_DIR": CONDA_DIR,
    "ENV_DIR": ENV_DIR,
    "PYTHON": PYTHON,
    "BASE_YAML_CONFIG": BASE_YAML_CONFIG,
    "RUNTIME_YAML_CONFIG": RUNTIME_YAML_CONFIG,
    "YAML_CONFIG": YAML_CONFIG,
    "NOTEBOOK_PATH": NOTEBOOK_PATH,
    "OUTPUT_DIR": OUTPUT_DIR,
    "RUN_UUID": RUN_UUID,
    "MLFLOW_TRACKING_URI": f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow",
    "MLFLOW_TRACKING_USERNAME": DAGSHUB_USERNAME,
    "MLFLOW_TRACKING_PASSWORD": DAGSHUB_TOKEN,
    "MLFLOW_EXPERIMENT_NAME": EXPERIMENT_NAME,
    "MLFLOW_FLATTEN_PARAMS": "TRUE",
    "MLFLOW_TAGS": json.dumps({
        "run_uuid": RUN_UUID,
        "source": "kaggle",
        "task": "sft",
        "dataset": "parquet_crohme_train",
    }),
})

print(f"Run UUID: {RUN_UUID}")
print(f"MLflow: {os.environ['MLFLOW_TRACKING_URI']}")
print(f"PROJECT_DIR: {PROJECT_DIR}")
print(f"PROJECT_DIR in sys.path: {PROJECT_DIR in sys.path}")
print(f"PYTHONPATH: {os.environ.get('PYTHONPATH')}")
print(f"runtime_yaml.py exists: {Path(PROJECT_DIR, 'scripts/runtime_yaml.py').exists()}")

Run UUID: 2827040d00cb4ff88c57db6ee3a257e6
MLflow: https://dagshub.com/NhatPot/test-unimer.mlflow
PROJECT_DIR: /kaggle/working/test-unimer
PROJECT_DIR in sys.path: True
PYTHONPATH: /kaggle/working/test-unimer
runtime_yaml.py exists: False


In [2]:
%%bash
# 2. Tạo môi trường Python 3.10
set -euo pipefail

if [[ ! -x "$PYTHON" ]]; then
  wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
  bash /tmp/miniconda.sh -b -f -p "$CONDA_DIR"
  rm -f /tmp/miniconda.sh
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
  "$CONDA_DIR/bin/conda" tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r
  "$CONDA_DIR/bin/conda" create -n unimumer python=3.10 -y
fi

"$PYTHON" --version

PREFIX=/kaggle/working/miniconda
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
    You currently have a PYTHONPATH environment variable set. This may cause
    unexpected behavior when running the Python interpreter in Miniconda3.
    For best results, please verify that your PYTHONPATH only points to
    directories of packages that are compatible with the Python interpreter
    in Miniconda3: /kaggle/working/miniconda
accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: done
Channels:
 - defaults
Platform: linux-64
Solving environment: done

## Package Plan ##

  environment location: /kaggle/working/miniconda/envs/unimumer

  added / updated specs:
    - python=3.10


The following packages will



==> WARNING: A newer version of conda exists. <==
    current version: 26.3.2
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda




In [3]:
%%bash
# 3. Lấy source code
set -euo pipefail

if [[ ! -d "$PROJECT_DIR/.git" ]]; then
  git clone https://github.com/NhatPot/test-unimer.git "$PROJECT_DIR"
fi

git -C "$PROJECT_DIR" rev-parse --short HEAD

3751f53


Cloning into '/kaggle/working/test-unimer'...


In [4]:
%%bash
# 4. Cài dependency
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python --version
python -m pip install -q -r requirements.txt
python -m pip install -q -e train/LLaMA-Factory
python -c "import torch, mlflow; print('GPU:', torch.cuda.get_device_name(0)); print('MLflow:', mlflow.__version__)"

Python 3.10.20
GPU: Tesla T4
MLflow: 3.14.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
prometheus-fastapi-instrumentator 8.0.2 requires starlette<2.0.0,>=1.0.0, but you have starlette 0.52.1 which is incompatible.


In [5]:
# 5. Runtime YAML Override (tùy chọn)
from scripts.runtime_yaml import prepare_runtime_yaml

USE_RUNTIME_YAML_OVERRIDE = True

YAML_OVERRIDES = {
    "dataset": "parquet_crohme_train",
    "max_samples": 3,
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 64,
    "learning_rate": 1.0e-4,
    "lora_rank": 64,
    "logging_steps": 1,
    "save_steps": 20,
    "output_dir": "saves/qwen2.5_vl-3b/qlora/sft/standred/uni-mumer_qlora",
    
    # Để None nếu không muốn override
    "cutoff_len": None,
    "val_size": None,
    "per_device_eval_batch_size": None,
    "eval_steps": None,
    "save_total_limit": None,
    "lora_alpha": None,
    "lora_dropout": None,
    "warmup_ratio": None,
    "quantization_bit": None,
    "preprocessing_num_workers": None,
    "dataloader_num_workers": None,
    "bf16": None,
    "fp16": None,
}

YAML_CONFIG, OUTPUT_DIR, YAML_DATA = prepare_runtime_yaml(
    project_dir=PROJECT_DIR,
    base_yaml_config=BASE_YAML_CONFIG,
    runtime_yaml_config=RUNTIME_YAML_CONFIG,
    use_override=USE_RUNTIME_YAML_OVERRIDE,
    overrides=YAML_OVERRIDES,
    strict_keys=True,
)

# Cập nhật biến môi trường
mlflow_tags = json.loads(os.environ["MLFLOW_TAGS"])
mlflow_tags.update({
    "dataset": str(YAML_DATA.get("dataset", "")),
    "yaml_config": YAML_CONFIG,
    "yaml_override": str(USE_RUNTIME_YAML_OVERRIDE).lower(),
})

os.environ.update({
    "YAML_CONFIG": YAML_CONFIG,
    "OUTPUT_DIR": OUTPUT_DIR,
    "MLFLOW_TAGS": json.dumps(mlflow_tags),
})

Runtime YAML Override: ON
YAML gốc: /kaggle/working/test-unimer/train/Uni-MuMER-train.yaml
YAML dùng để train: /kaggle/working/runtime_Uni-MuMER-train.yaml
Các key đã đổi:
  max_samples: 2 -> 3


In [6]:
%%bash
# 6. Kiểm tra DagsHub trước khi train
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

python scripts/dagshub_logger.py check --experiment "$MLFLOW_EXPERIMENT_NAME"

🏃 View run fearless-jay-151 at: https://dagshub.com/NhatPot/test-unimer.mlflow/#/experiments/1/runs/a8525d81e0124149b2f0723050cb8d85
🧪 View experiment at: https://dagshub.com/NhatPot/test-unimer.mlflow/#/experiments/1
DagsHub MLflow connection OK: https://dagshub.com/NhatPot/test-unimer.mlflow


In [7]:
%%bash
# 7. Training
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

echo "Python: $(command -v python)"
echo "Torchrun: $(command -v torchrun)"
MPLBACKEND=Agg llamafactory-cli train "$YAML_CONFIG" "run_name=uni-mumer-${RUN_UUID:0:8}"

Python: /kaggle/working/miniconda/envs/unimumer/bin/python
Torchrun: /kaggle/working/miniconda/envs/unimumer/bin/torchrun
[INFO|2026-06-24 02:59:13] llamafactory.launcher:143 >> Initializing 2 distributed tasks at: 127.0.0.1:52745
[WARNING|2026-06-24 02:59:22] llamafactory.hparams.parser:148 >> We recommend enable `upcast_layernorm` in quantized training.
[INFO|2026-06-24 02:59:22] llamafactory.hparams.parser:143 >> Set `ddp_find_unused_parameters` to False in DDP training since LoRA is enabled.
[INFO|2026-06-24 02:59:22] llamafactory.hparams.parser:465 >> Process rank: 0, world size: 2, device: cuda:0, distributed training: True, compute dtype: torch.float16
[INFO|2026-06-24 02:59:23] llamafactory.hparams.parser:465 >> Process rank: 1, world size: 2, device: cuda:1, distributed training: True, compute dtype: torch.float16
[INFO|2026-06-24 02:59:26] llamafactory.data.loader:143 >> Loading dataset phxember/Uni-MuMER-Data...
training example:
input_ids:
[151644, 8948, 198, 2610, 525, 264

W0624 02:59:14.887000 403 site-packages/torch/distributed/run.py:792] 
W0624 02:59:14.887000 403 site-packages/torch/distributed/run.py:792] *****************************************
W0624 02:59:14.887000 403 site-packages/torch/distributed/run.py:792] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0624 02:59:14.887000 403 site-packages/torch/distributed/run.py:792] *****************************************
[INFO|tokenization_utils_base.py:2023] 2026-06-24 02:59:23,656 >> loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct/snapshots/66285546d2b821cf421d4f5eb2576359d3770cd3/vocab.json
[INFO|tokenization_utils_base.py:2023] 2026-06-24 02:59:23,656 >> loading file merges.txt from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-VL-3B-Instruct/snapshots/66285546d2b82

In [ ]:
%%bash
# 9. Upload artifacts và test results lên DagsHub
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

# Upload model checkpoint và config
python scripts/dagshub_logger.py upload \
  --experiment "$MLFLOW_EXPERIMENT_NAME" \
  --run-uuid "$RUN_UUID" \
  --config "$YAML_CONFIG" \
  --output-dir "$OUTPUT_DIR" \
  --project-dir "$PROJECT_DIR" \
  --notebook "$NOTEBOOK_PATH"

# Upload test results (nếu có)
if [[ -d "kaggle_test_results" ]]; then
  echo ""
  echo "Uploading test results to DagsHub..."
  
  # Tạo archive của test results
  tar -czf kaggle_test_results.tar.gz kaggle_test_results/
  
  python scripts/dagshub_logger.py upload \
    --experiment "$MLFLOW_EXPERIMENT_NAME" \
    --run-uuid "$RUN_UUID" \
    --artifact-path kaggle_test_results.tar.gz \
    --artifact-type "test_results"
  
  echo "✓ Test results uploaded"
fi

In [ ]:
%%bash
# 8. Full Benchmark Test (3 CROHME datasets)
set -euo pipefail
cd "$PROJECT_DIR"
source "$CONDA_DIR/bin/activate" unimumer

# Tìm checkpoint cuối cùng
LAST_CKPT=$(ls -td "$OUTPUT_DIR"/checkpoint-* 2>/dev/null | head -1)

if [[ -z "$LAST_CKPT" ]]; then
  echo "ERROR: No checkpoint found in $OUTPUT_DIR"
  exit 1
fi

echo "Testing with checkpoint: $LAST_CKPT"
echo "Running full benchmark on 3 CROHME datasets (3,332 samples)..."
echo ""

# Chạy full benchmark
python scripts/kaggle_full_test.py \
  --base-model Qwen/Qwen2.5-VL-3B-Instruct \
  --adapter-path "$LAST_CKPT" \
  --test-datasets crohme_2014 crohme_2016 crohme_2019 \
  --backup-dir example_data/backup \
  --base-results-dir example_data/CROHME/results \
  --output-dir kaggle_test_results \
  --project-dir "$PROJECT_DIR" \
  --batch-size 2

# In summary
echo ""
echo "============================================================"
echo "                    TEST SUMMARY"
echo "============================================================"
for dataset in crohme_2014 crohme_2016 crohme_2019; do
  echo ""
  echo "=== $dataset ==="
  if [[ -f "kaggle_test_results/${dataset}_results.txt" ]]; then
    cat "kaggle_test_results/${dataset}_results.txt" | grep -E "(Mean Edit Score|BLEU-4|Character Error Rate|Exact Match)" | head -4
  else
    echo "Results not found"
  fi
done

echo ""
echo "============================================================"
echo "Full comparison table:"
cat kaggle_test_results/comparison_table.txt